# Word2Vec Antonym Space Exploration

This notebook allows interactive exploration of the antonym space.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
from antonym_loader import extract_antonym_pairs, group_antonyms_by_word
from word2vec_loader import Word2VecLoader
from antonym_space import create_antonym_space_from_model
from visualization import (
    plot_antonym_axes_2d,
    plot_near_zero_words,
    plot_word_profile,
    plot_antonym_pair_comparison,
    plot_coverage_breakdown
)

## 1. Load Data and Model

In [ ]:
# Load antonym pairs from WordNet
antonym_pairs = extract_antonym_pairs()
known_antonyms = group_antonyms_by_word()
print(f"Loaded {len(antonym_pairs)} antonym pairs")

In [ ]:
# Load Word2Vec model (GloVe-100 is a good balance of speed and quality)
loader = Word2VecLoader()
model = loader.load_glove(100)  # Use 50, 100, 200, or 300

## 2. Build Antonym Space

In [ ]:
# Filter to valid pairs (both words in vocabulary)
valid_pairs = [
    (w1, w2) for w1, w2 in antonym_pairs
    if loader.has_word(w1) and loader.has_word(w2)
]
print(f"Valid pairs: {len(valid_pairs)} / {len(antonym_pairs)}")

In [ ]:
# Build the antonym space using greedy orthogonalization
space = create_antonym_space_from_model(
    model,
    valid_pairs,
    method='greedy',  # or 'svd'
    max_axes=100,
    min_orthogonality=0.1
)
print(f"Created {len(space.axes)} axes")

In [ ]:
# View the axes
for i, axis in enumerate(space.axes[:20]):
    print(f"Axis {i:2d}: {axis.negative_word:15s} <-> {axis.positive_word:15s} (sep={axis.separation:.3f})")

## 3. Find Words Near Zero (Semantic Neutrals)

In [ ]:
# Find words closest to the zero vector
near_zero = space.find_near_zero_words(top_n=100)

print("Top 30 semantically neutral words:")
for word, dist, _ in near_zero[:30]:
    print(f"  {word:20s}: {dist:.4f}")

In [ ]:
# Visualize near-zero words
plot_near_zero_words(near_zero, space, n_words=50)

## 4. Discover Potential Antonyms

In [ ]:
# Find potential antonyms for a word
word = "technology"
potential_antonyms = space.find_potential_antonyms(word, top_n=10)

print(f"Potential antonyms for '{word}':")
for ant, score in potential_antonyms:
    print(f"  {ant}: {score:.3f}")

In [ ]:
# Try more words
test_words = ["computer", "science", "music", "democracy", "philosophy"]

for word in test_words:
    potentials = space.find_potential_antonyms(word, top_n=5)
    if potentials:
        ant_str = ", ".join([f"{a}({s:.2f})" for a, s in potentials[:3]])
        print(f"{word}: {ant_str}")

## 5. Analyze Word Profiles

In [ ]:
# View a word's semantic profile
plot_word_profile("happy", space, top_n_axes=15)

In [ ]:
# Compare two words
plot_antonym_pair_comparison("love", "hate", space, top_n_axes=10)

## 6. Coverage Analysis

In [ ]:
# Analyze how much of the semantic space is captured
coverage = space.analyze_axis_coverage()
print(f"Explained variance ratio: {coverage['explained_ratio']:.4f}")
print(f"Number of axes: {coverage['num_axes']}")

In [ ]:
# Visualize coverage breakdown
plot_coverage_breakdown(space)

## 7. 2D Visualization

In [ ]:
# Plot words on 2D plane of two antonym axes
sample_words = ["happy", "sad", "love", "hate", "good", "bad", 
                "light", "dark", "big", "small", "hot", "cold",
                "fast", "slow", "young", "old", "rich", "poor"]

plot_antonym_axes_2d(space, sample_words, axes_to_show=(0, 1))